In [1]:
# =============================================================================
# 01_boundary_QA.ipynb  (Notebook-friendly, production-style QA for ADM polygons)
# =============================================================================
# Purpose:
# - Run a broad QA checklist on a polygon boundary layer used for zonal statistics
# - Produce:
#   (1) concise console summary
#   (2) outputs:
#       - qa_summary.json  (structured results)
#       - qa_summary.csv   (one-row-per-check)
#       - issues_features.gpkg (only flagged features; preferred)
#       - optional extra CSVs if relevant
#
# Dependencies (as requested): geopandas, shapely, pyproj, pandas, numpy, fiona, stdlib
# =============================================================================

from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon
from shapely.validation import explain_validity

# Try to import make_valid if available (Shapely 2.x)
try:
    from shapely.validation import make_valid  # type: ignore
    HAS_MAKE_VALID = True
except Exception:
    HAS_MAKE_VALID = False


# -----------------------------------------------------------------------------
# USER SETTINGS
# -----------------------------------------------------------------------------
boundary_path = r"C:\Users\am636\Desktop\Job\applied\Shortlisted\Specialist Technician_Demographic Data\Interview_package2\in_phyton\data_raw\ZMB_adm2_gadm41_Copperbelt.shp"
output_dir   = r"C:\Users\am636\Desktop\Job\applied\Shortlisted\Specialist Technician_Demographic Data\Interview_package2\in_phyton\outputs"

# Optional: expected province/ADM1 name (set to None if you don't want to enforce)
expected_name1 = "Copperbelt"   # or None

# Equal-area CRS for area-based diagnostics (global, safe for comparisons)
equal_area_crs = "EPSG:6933"  # World Cylindrical Equal Area


# -----------------------------------------------------------------------------
# Paths + run metadata
# -----------------------------------------------------------------------------
boundary_path = Path(boundary_path)
output_dir = Path(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

run_time_utc = datetime.now(timezone.utc).isoformat(timespec="seconds")

# Output filenames
json_out = output_dir / "qa_summary.json"
csv_out  = output_dir / "qa_summary.csv"
gpkg_out = output_dir / "issues_features.gpkg"

# Optional CSVs (write only if relevant)
dup_id_csv    = output_dir / "duplicates_by_id.csv"
dup_geom_csv  = output_dir / "duplicates_by_geometry.csv"
invalid_csv   = output_dir / "invalid_geometries.csv"
outscope_csv  = output_dir / "out_of_scope.csv"
overlaps_csv  = output_dir / "overlaps.csv"
gaps_csv      = output_dir / "gaps.csv"
slivers_csv   = output_dir / "slivers.csv"


# -----------------------------------------------------------------------------
# Helpers: QA recording + statuses
# -----------------------------------------------------------------------------
def status_rank(s):
    return {"PASS": 0, "WARN": 1, "FAIL": 2}.get(s, 1)

def worst_status(statuses):
    return max(statuses, key=status_rank) if statuses else "PASS"

checks = []  # one dict per check (for CSV + JSON)

def add_check(name, status, metrics=None, note=""):
    if metrics is None:
        metrics = {}
    checks.append({
        "check": name,
        "status": status,
        "metrics": metrics,
        "note": note
    })

def safe_str(x):
    try:
        return str(x)
    except Exception:
        return repr(x)

def has_weird_encoding(series: pd.Series) -> bool:
    # Very lightweight heuristic: look for replacement char or obvious corruption
    if series.dtype != "object":
        return False
    s = series.dropna().astype(str)
    if s.empty:
        return False
    bad = s.str.contains("�", regex=False).any()
    return bool(bad)

def geom_vertex_count(geom):
    if geom is None or geom.is_empty:
        return 0
    gt = geom.geom_type
    if gt == "Polygon":
        # exterior + interiors
        n = len(geom.exterior.coords) if geom.exterior else 0
        for ring in geom.interiors:
            n += len(ring.coords)
        return n
    if gt == "MultiPolygon":
        return sum(geom_vertex_count(g) for g in geom.geoms)
    # unexpected
    return 0

def geometry_hash_md5_wkb(geom):
    if geom is None or geom.is_empty:
        return None
    # WKB is stable for exact duplicates within the same environment
    h = hashlib.md5(geom.wkb).hexdigest()
    return h

def try_fix_geometry(geom):
    """
    Attempt to fix invalid geometry.
    - Prefer make_valid if available.
    - Fallback: buffer(0) (classic repair for some self-intersections).
    Returns (fixed_geom, method_used or None)
    """
    if geom is None or geom.is_empty:
        return geom, None
    if geom.is_valid:
        return geom, None

    if HAS_MAKE_VALID:
        try:
            g2 = make_valid(geom)
            if g2 is not None and (not g2.is_empty) and g2.is_valid:
                return g2, "make_valid"
        except Exception:
            pass

    # buffer(0) fallback
    try:
        g2 = geom.buffer(0)
        if g2 is not None and (not g2.is_empty) and g2.is_valid:
            return g2, "buffer(0)"
    except Exception:
        pass

    return geom, None

def union_hole_areas(union_geom):
    """
    Detect holes inside a union geometry. Holes can indicate internal gaps (uncovered areas).
    Returns:
      - hole_count
      - total_hole_area (in the geometry's CRS units^2; use equal-area for meaningful areas)
    """
    if union_geom is None or union_geom.is_empty:
        return 0, 0.0

    hole_count = 0
    hole_area = 0.0

    def poly_holes(poly: Polygon):
        nonlocal hole_count, hole_area
        for ring in poly.interiors:
            hole_count += 1
            hole_area += Polygon(ring).area

    gt = union_geom.geom_type
    if gt == "Polygon":
        poly_holes(union_geom)
    elif gt == "MultiPolygon":
        for p in union_geom.geoms:
            poly_holes(p)

    return hole_count, float(hole_area)


# -----------------------------------------------------------------------------
# A) File + read checks
# -----------------------------------------------------------------------------
if not boundary_path.exists():
    add_check(
        "A1 File exists",
        "FAIL",
        metrics={"path": str(boundary_path)},
        note="Boundary file does not exist."
    )
    # Write minimal outputs and stop.
    pd.DataFrame(checks).to_csv(csv_out, index=False)
    with open(json_out, "w", encoding="utf-8") as f:
        json.dump({"run_time_utc": run_time_utc, "checks": checks}, f, indent=2)
    raise FileNotFoundError(boundary_path)

add_check("A1 File exists", "PASS", metrics={"path": str(boundary_path)}, note="Boundary file exists.")

try:
    gdf = gpd.read_file(boundary_path)
    add_check("A2 Readable layer", "PASS", metrics={"features": int(len(gdf))}, note="Layer read successfully.")
except Exception as e:
    add_check("A2 Readable layer", "FAIL", metrics={"error": safe_str(e)}, note="Failed to read layer.")
    pd.DataFrame(checks).to_csv(csv_out, index=False)
    with open(json_out, "w", encoding="utf-8") as f:
        json.dump({"run_time_utc": run_time_utc, "checks": checks}, f, indent=2)
    raise

# Geometry type sanity
geom_types = sorted(set(gdf.geom_type.dropna().unique().tolist()))
unexpected = [t for t in geom_types if t not in ["Polygon", "MultiPolygon"]]
if unexpected:
    add_check(
        "A3 Geometry types are polygon-only",
        "FAIL",
        metrics={"geom_types": geom_types, "unexpected": unexpected},
        note="Non-polygon geometries present; not suitable for polygon zonal stats without fixing."
    )
else:
    add_check(
        "A3 Geometry types are polygon-only",
        "PASS",
        metrics={"geom_types": geom_types},
        note="Only Polygon/MultiPolygon present."
    )

# Encoding heuristic
obj_cols = [c for c in gdf.columns if gdf[c].dtype == "object" and c != "geometry"]
encoding_flags = {c: has_weird_encoding(gdf[c]) for c in obj_cols}
bad_cols = [c for c, bad in encoding_flags.items() if bad]
if bad_cols:
    add_check(
        "A4 Encoding / text corruption heuristic",
        "WARN",
        metrics={"flagged_columns": bad_cols},
        note="Some text fields contain replacement characters (�). Consider verifying encoding or re-exporting."
    )
else:
    add_check(
        "A4 Encoding / text corruption heuristic",
        "PASS",
        metrics={"checked_object_columns": len(obj_cols)},
        note="No obvious text corruption detected (heuristic)."
    )


# -----------------------------------------------------------------------------
# B) CRS checks
# -----------------------------------------------------------------------------
crs = gdf.crs
if crs is None:
    add_check(
        "B1 CRS exists and parseable",
        "FAIL",
        metrics={"crs": None},
        note="CRS is missing. You must define it before any spatial operations."
    )
    crs_is_geographic = None
    crs_epsg = None
else:
    try:
        crs_epsg = crs.to_epsg()
    except Exception:
        crs_epsg = None
    add_check(
        "B1 CRS exists and parseable",
        "PASS",
        metrics={"crs": safe_str(crs), "epsg": crs_epsg},
        note="CRS found."
    )
    crs_is_geographic = bool(crs.is_geographic)

if crs is not None and crs_is_geographic:
    add_check(
        "B2 CRS is geographic note",
        "WARN",
        metrics={"is_geographic": True},
        note=f"Layer CRS is geographic (degrees). Area/length are not meaningful unless projected (e.g., {equal_area_crs})."
    )
elif crs is not None:
    add_check(
        "B2 CRS is geographic note",
        "PASS",
        metrics={"is_geographic": False},
        note="Layer CRS is projected; area/length are meaningful in CRS units."
    )


# -----------------------------------------------------------------------------
# C) Schema / attribute checks
# -----------------------------------------------------------------------------
# Columns + dtypes
col_dtypes = {c: safe_str(gdf[c].dtype) for c in gdf.columns if c != "geometry"}
add_check(
    "C1 Schema overview",
    "PASS",
    metrics={"n_columns": len(gdf.columns), "columns": list(gdf.columns), "dtypes": col_dtypes},
    note="Schema recorded."
)

# Likely key fields
likely_keys = ["GID_0", "GID_1", "GID_2", "NAME_1", "NAME_2"]
missing_keys = [k for k in likely_keys if k not in gdf.columns]
if missing_keys:
    add_check(
        "C2 Key fields present (soft check)",
        "WARN",
        metrics={"missing": missing_keys, "present": [k for k in likely_keys if k in gdf.columns]},
        note="Some common admin fields are missing; QA will adapt."
    )
else:
    add_check(
        "C2 Key fields present (soft check)",
        "PASS",
        metrics={"present": likely_keys},
        note="All common admin fields are present."
    )

# Null/empty counts for key fields (or all fields if easy)
null_stats = {}
for c in [k for k in likely_keys if k in gdf.columns]:
    s = gdf[c]
    n_null = int(s.isna().sum())
    n_empty = int((s.astype(str).str.strip() == "").sum()) if s.dtype == "object" else 0
    null_stats[c] = {"null": n_null, "empty_string": n_empty}

if null_stats:
    any_problem = any(v["null"] > 0 or v["empty_string"] > 0 for v in null_stats.values())
    add_check(
        "C3 Null/empty counts for key fields",
        "WARN" if any_problem else "PASS",
        metrics=null_stats,
        note="Key field completeness checked."
    )
else:
    add_check(
        "C3 Null/empty counts for key fields",
        "WARN",
        metrics={},
        note="No key fields available to check completeness."
    )

# Uniqueness check on GID_2 if exists
if "GID_2" in gdf.columns:
    gid2 = gdf["GID_2"].astype(str)
    dup_mask = gid2.duplicated(keep=False)
    n_dup = int(dup_mask.sum())
    if n_dup > 0:
        dup_vals = sorted(gid2[dup_mask].unique().tolist())
        add_check(
            "C4 Uniqueness of GID_2",
            "FAIL",
            metrics={"duplicates_count": n_dup, "duplicate_values": dup_vals[:50], "duplicate_values_truncated": len(dup_vals) > 50},
            note="Duplicate GID_2 values found."
        )
        # Save duplicates_by_id.csv
        dup_df = gdf.loc[dup_mask, ["GID_2"] + ([c for c in ["NAME_1", "NAME_2"] if c in gdf.columns])]
        dup_df = dup_df.assign(row_index=dup_df.index.values)
        dup_df.to_csv(dup_id_csv, index=False)
    else:
        add_check(
            "C4 Uniqueness of GID_2",
            "PASS",
            metrics={"duplicates_count": 0},
            note="PASS: no duplicate GID_2 values found."
        )
else:
    add_check(
        "C4 Uniqueness of stable ID",
        "WARN",
        metrics={},
        note="No GID_2 field found. Consider creating a surrogate ID for stable joins."
    )

# Hierarchy consistency: NAME_2 -> NAME_1 mapping and single-province expectation
hier_notes = []
hier_statuses = []

if "NAME_1" in gdf.columns:
    uniq_name1 = sorted(gdf["NAME_1"].dropna().astype(str).unique().tolist())
    if len(uniq_name1) <= 1:
        add_check(
            "C5 Province consistency (unique NAME_1 values)",
            "PASS",
            metrics={"unique_NAME_1": uniq_name1},
            note="Single NAME_1 value (as expected for a province subset)."
        )
    else:
        add_check(
            "C5 Province consistency (unique NAME_1 values)",
            "WARN",
            metrics={"unique_NAME_1": uniq_name1},
            note="More than one NAME_1 value found; may indicate out-of-scope features or mixed provinces."
        )

    if expected_name1:
        mismatch = gdf["NAME_1"].astype(str) != expected_name1
        n_mismatch = int(mismatch.sum())
        if n_mismatch > 0:
            add_check(
                "G1 Out-of-scope by expected NAME_1",
                "FAIL",
                metrics={"expected_NAME_1": expected_name1, "mismatch_count": n_mismatch},
                note="Features not matching expected province name detected."
            )
            # Save out_of_scope.csv
            cols = [c for c in ["GID_2", "NAME_1", "NAME_2"] if c in gdf.columns]
            out_df = gdf.loc[mismatch, cols].copy()
            out_df["row_index"] = out_df.index.values
            out_df.to_csv(outscope_csv, index=False)
        else:
            add_check(
                "G1 Out-of-scope by expected NAME_1",
                "PASS",
                metrics={"expected_NAME_1": expected_name1, "mismatch_count": 0},
                note="PASS: all features match expected province name."
            )
else:
    add_check(
        "C5 Province consistency (unique NAME_1 values)",
        "WARN",
        metrics={},
        note="NAME_1 not present; cannot check province consistency or expected province name."
    )
    if expected_name1:
        add_check(
            "G1 Out-of-scope by expected NAME_1",
            "WARN",
            metrics={"expected_NAME_1": expected_name1},
            note="expected_name1 provided but NAME_1 field is missing; cannot enforce out-of-scope check."
        )

if "NAME_1" in gdf.columns and "NAME_2" in gdf.columns:
    # Check if same NAME_2 maps to multiple NAME_1 values
    mapping = gdf.groupby(gdf["NAME_2"].astype(str))["NAME_1"].nunique(dropna=True)
    bad = mapping[mapping > 1]
    if len(bad) > 0:
        add_check(
            "C6 Hierarchy: NAME_2 maps cleanly to a single NAME_1",
            "WARN",
            metrics={"problematic_NAME_2_count": int(len(bad)), "examples": bad.head(10).to_dict()},
            note="Some district names appear under multiple provinces; may indicate mixed datasets or naming ambiguity."
        )
    else:
        add_check(
            "C6 Hierarchy: NAME_2 maps cleanly to a single NAME_1",
            "PASS",
            metrics={"problematic_NAME_2_count": 0},
            note="PASS: NAME_2 values map to a single NAME_1."
        )
else:
    add_check(
        "C6 Hierarchy: NAME_2 maps cleanly to a single NAME_1",
        "WARN",
        metrics={},
        note="NAME_1 and/or NAME_2 missing; cannot check hierarchy consistency."
    )


# -----------------------------------------------------------------------------
# D) Geometry validity checks
# -----------------------------------------------------------------------------
geom = gdf.geometry

n_missing_geom = int(geom.isna().sum())
n_empty_geom = int(geom.is_empty.sum()) if hasattr(geom, "is_empty") else 0
valid_mask = geom.is_valid.fillna(False)

n_invalid = int((~valid_mask).sum()) - n_missing_geom  # approximate; missing counted separately
invalid_idx = gdf.index[(~valid_mask) & (~geom.isna())].tolist()

if n_missing_geom > 0 or n_empty_geom > 0 or n_invalid > 0:
    status = "FAIL" if (n_missing_geom > 0 or n_empty_geom > 0) else "WARN"
    add_check(
        "D1 Missing/empty/invalid geometries",
        status,
        metrics={"missing_geom": n_missing_geom, "empty_geom": n_empty_geom, "invalid_geom": int(n_invalid)},
        note="Geometry validity problems found."
    )
else:
    add_check(
        "D1 Missing/empty/invalid geometries",
        "PASS",
        metrics={"missing_geom": 0, "empty_geom": 0, "invalid_geom": 0},
        note="PASS: no missing, empty, or invalid geometries detected."
    )

# Explain invalidity + attempt fixability estimate
if n_invalid > 0:
    invalid_rows = []
    fixable = 0
    fix_methods = {"make_valid": 0, "buffer(0)": 0}

    for idx in invalid_idx:
        g = gdf.at[idx, "geometry"]
        reason = explain_validity(g)
        g2, method = try_fix_geometry(g)
        can_fix = bool(g2 is not None and (not g2.is_empty) and g2.is_valid)
        if can_fix:
            fixable += 1
            if method:
                fix_methods[method] = fix_methods.get(method, 0) + 1

        rec = {
            "row_index": idx,
            "reason": reason,
            "fixable": can_fix,
            "fix_method": method
        }
        for c in ["GID_2", "NAME_2", "NAME_1"]:
            if c in gdf.columns:
                rec[c] = gdf.at[idx, c]
        invalid_rows.append(rec)

    inv_df = pd.DataFrame(invalid_rows)
    inv_df.to_csv(invalid_csv, index=False)

    add_check(
        "D2 Invalid geometry diagnostics + fixability",
        "WARN" if fixable > 0 else "FAIL",
        metrics={"invalid_count": int(n_invalid), "fixable_estimate": int(fixable), "fix_methods": fix_methods, "saved_csv": str(invalid_csv)},
        note=("Some invalid geometries appear fixable. In production: repair (make_valid/buffer), re-validate, and document changes."
              if fixable > 0 else "Invalid geometries detected and not trivially fixable; manual inspection likely required.")
    )
else:
    add_check(
        "D2 Invalid geometry diagnostics + fixability",
        "PASS",
        metrics={"invalid_count": 0},
        note="PASS: no invalid geometries, so no fixability analysis needed."
    )


# -----------------------------------------------------------------------------
# E) Duplicate geometry checks (hash of WKB)
# -----------------------------------------------------------------------------
geom_hash = []
for g in gdf.geometry:
    geom_hash.append(geometry_hash_md5_wkb(g))

gdf["_geom_hash"] = geom_hash

dup_geom_mask = gdf["_geom_hash"].notna() & gdf["_geom_hash"].duplicated(keep=False)
n_dup_geom = int(dup_geom_mask.sum())

if n_dup_geom > 0:
    # Group duplicates
    groups = (gdf.loc[dup_geom_mask]
              .groupby("_geom_hash")
              .apply(lambda x: x.index.tolist())
              .to_dict())

    add_check(
        "E1 Duplicate geometries (exact)",
        "FAIL",
        metrics={"duplicate_groups": int(len(groups)), "features_involved": int(n_dup_geom)},
        note="Exact duplicate geometries detected (by WKB hash)."
    )

    # Save details CSV
    rows = []
    for h, idxs in groups.items():
        for idx in idxs:
            rec = {"geom_hash": h, "row_index": idx}
            for c in ["GID_2", "NAME_2", "NAME_1"]:
                if c in gdf.columns:
                    rec[c] = gdf.at[idx, c]
            rows.append(rec)
    pd.DataFrame(rows).to_csv(dup_geom_csv, index=False)
else:
    add_check(
        "E1 Duplicate geometries (exact)",
        "PASS",
        metrics={"duplicate_groups": 0, "features_involved": 0},
        note="PASS: no exact duplicate geometries detected."
    )


# -----------------------------------------------------------------------------
# F) Topology checks (overlaps, gaps-as-holes, slivers)
# -----------------------------------------------------------------------------
# Use projected geometries for area-based metrics
if gdf.crs is not None:
    try:
        gdf_ea = gdf.to_crs(equal_area_crs)
        ea_ok = True
        add_check(
            "F0 Equal-area projection for diagnostics",
            "PASS",
            metrics={"equal_area_crs": equal_area_crs},
            note="Reprojected to equal-area CRS for area diagnostics."
        )
    except Exception as e:
        ea_ok = False
        gdf_ea = None
        add_check(
            "F0 Equal-area projection for diagnostics",
            "WARN",
            metrics={"equal_area_crs": equal_area_crs, "error": safe_str(e)},
            note="Could not reproject to equal-area CRS. Area-based diagnostics may be limited."
        )
else:
    ea_ok = False
    gdf_ea = None
    add_check(
        "F0 Equal-area projection for diagnostics",
        "WARN",
        metrics={"equal_area_crs": equal_area_crs},
        note="Missing CRS prevents projection to equal-area; area-based diagnostics limited."
    )

# F1 Overlaps: use spatial index to avoid O(n^2)
# Overlap = intersects but not just touching; quantify overlap area in equal-area CRS if possible
overlap_pairs = []
try:
    sindex = gdf.sindex
    idx_list = list(gdf.index)
    for i in idx_list:
        geom_i = gdf.at[i, "geometry"]
        if geom_i is None or geom_i.is_empty:
            continue
        cand = list(sindex.intersection(geom_i.bounds))
        for j in cand:
            if j <= i:
                continue
            geom_j = gdf.at[j, "geometry"]
            if geom_j is None or geom_j.is_empty:
                continue
            if not geom_i.intersects(geom_j):
                continue
            if geom_i.touches(geom_j):
                continue  # touching is ok
            # true overlap
            rec = {"i": i, "j": j}
            for c in ["GID_2", "NAME_2"]:
                if c in gdf.columns:
                    rec[f"{c}_i"] = gdf.at[i, c]
                    rec[f"{c}_j"] = gdf.at[j, c]
            if ea_ok:
                gi = gdf_ea.at[i, "geometry"]
                gj = gdf_ea.at[j, "geometry"]
                inter_area = gi.intersection(gj).area
                rec["overlap_area_m2"] = float(inter_area)
            overlap_pairs.append(rec)
except Exception as e:
    add_check(
        "F1 Overlaps between polygons",
        "WARN",
        metrics={"error": safe_str(e)},
        note="Overlap check encountered an error. Consider running in QGIS for cross-check."
    )
    overlap_pairs = None

if overlap_pairs is not None:
    if len(overlap_pairs) > 0:
        add_check(
            "F1 Overlaps between polygons",
            "FAIL",
            metrics={"overlap_pair_count": int(len(overlap_pairs)), "saved_csv": str(overlaps_csv)},
            note="Overlapping polygon pairs detected. This can inflate/duplicate zonal results."
        )
        pd.DataFrame(overlap_pairs).to_csv(overlaps_csv, index=False)
    else:
        add_check(
            "F1 Overlaps between polygons",
            "PASS",
            metrics={"overlap_pair_count": 0},
            note="PASS: no overlaps detected (touching boundaries allowed)."
        )

# F2 Gaps (limited): detect HOLES inside union (enclosed gaps)
# Note: Without authoritative outer boundary, we cannot detect open boundary gaps reliably.
if ea_ok:
    union = gdf_ea.geometry.unary_union
    hole_count, hole_area = union_hole_areas(union)
    union_area = float(union.area) if union and (not union.is_empty) else 0.0
    hole_ratio = (hole_area / union_area) if union_area > 0 else 0.0

    # Heuristic thresholds (very cautious)
    # - If holes exist with non-trivial area, warn/fail.
    if hole_count == 0:
        add_check(
            "F2 Enclosed gaps (holes inside union)",
            "PASS",
            metrics={"hole_count": 0, "hole_area_m2": 0.0, "hole_area_ratio": 0.0},
            note="PASS: no enclosed holes detected in the union (no obvious internal gaps)."
        )
    else:
        status = "WARN" if hole_ratio < 0.001 else "FAIL"
        add_check(
            "F2 Enclosed gaps (holes inside union)",
            status,
            metrics={"hole_count": int(hole_count), "hole_area_m2": float(hole_area), "hole_area_ratio": float(hole_ratio), "saved_csv": str(gaps_csv)},
            note=("Enclosed holes detected in union. These may indicate internal gaps (uncovered areas) between polygons."
                  " Without an authoritative province boundary, only enclosed gaps can be detected reliably.")
        )
        # Save a small CSV summary for holes (no per-feature attribution possible from union)
        pd.DataFrame([{
            "hole_count": hole_count,
            "hole_area_m2": hole_area,
            "hole_area_ratio": hole_ratio,
            "note": "Holes are properties of the union; they are not attributed to a single feature."
        }]).to_csv(gaps_csv, index=False)
else:
    add_check(
        "F2 Enclosed gaps (holes inside union)",
        "WARN",
        metrics={},
        note="Gap detection limited because equal-area projection failed or CRS missing. Consider checking in QGIS."
    )

# F3 Slivers: extremely small areas can indicate artifacts
if ea_ok:
    areas = gdf_ea.geometry.area
    # Robust threshold: below 0.5% of median area OR below 1 km^2 (1e6 m2) whichever is smaller
    median_area = float(np.nanmedian(areas.values)) if len(areas) else 0.0
    thresh = min(0.005 * median_area, 1e6) if median_area > 0 else 0.0
    sliver_mask = areas < thresh if thresh > 0 else np.array([False] * len(gdf_ea))
    n_slivers = int(np.sum(sliver_mask))

    if n_slivers > 0:
        add_check(
            "F3 Slivers (very small polygons)",
            "WARN",
            metrics={"sliver_count": n_slivers, "threshold_m2": float(thresh), "median_area_m2": median_area, "saved_csv": str(slivers_csv)},
            note="Very small polygons detected; may indicate artifacts or boundary slivers."
        )
        cols = [c for c in ["GID_2", "NAME_2", "NAME_1"] if c in gdf.columns]
        sl_df = gdf.loc[sliver_mask, cols].copy()
        sl_df["row_index"] = sl_df.index.values
        sl_df["area_m2"] = areas[sliver_mask].astype(float).values
        sl_df.to_csv(slivers_csv, index=False)
    else:
        add_check(
            "F3 Slivers (very small polygons)",
            "PASS",
            metrics={"sliver_count": 0, "threshold_m2": float(thresh), "median_area_m2": median_area},
            note="PASS: no slivers detected under the chosen threshold."
        )
else:
    add_check(
        "F3 Slivers (very small polygons)",
        "WARN",
        metrics={},
        note="Sliver detection requires equal-area projection; skipped/limited."
    )


# -----------------------------------------------------------------------------
# H) Practical “zonal statistics readiness” checks
# -----------------------------------------------------------------------------
# Complexity proxies
vertex_counts = gdf.geometry.apply(geom_vertex_count)
multipart_mask = gdf.geom_type == "MultiPolygon"
n_multipart = int(multipart_mask.sum())

complex_metrics = {
    "features": int(len(gdf)),
    "multipart_count": n_multipart,
    "vertex_count_min": int(vertex_counts.min()) if len(vertex_counts) else None,
    "vertex_count_median": float(np.median(vertex_counts.values)) if len(vertex_counts) else None,
    "vertex_count_p95": float(np.percentile(vertex_counts.values, 95)) if len(vertex_counts) else None,
    "vertex_count_max": int(vertex_counts.max()) if len(vertex_counts) else None,
}

# Warn if extremely complex (heuristic): p95 > 20k vertices
if len(vertex_counts) and complex_metrics["vertex_count_p95"] and complex_metrics["vertex_count_p95"] > 20000:
    add_check(
        "H1 Geometry complexity / multipart",
        "WARN",
        metrics=complex_metrics,
        note="Very high vertex counts detected. Zonal operations may be slow; consider simplifying for display only (not for analysis) or using faster engines."
    )
else:
    add_check(
        "H1 Geometry complexity / multipart",
        "PASS",
        metrics=complex_metrics,
        note="Complexity looks reasonable for zonal statistics (heuristic)."
    )

# Readiness suggestions (always recorded)
suggestions = [
    "Overlaps can double-count population in zonal sums; resolve overlaps or enforce a non-overlapping topology.",
    "Invalid/self-intersecting geometries can cause zonal tools to fail or return incorrect results; repair and re-validate before final reporting.",
    "Out-of-scope features can distort district rankings; either exclude them explicitly or report results with and without them."
]
add_check(
    "H2 Zonal statistics readiness notes",
    "PASS",
    metrics={"suggestions": suggestions},
    note="Practical notes recorded for reporting."
)


# -----------------------------------------------------------------------------
# Build a combined flagged-features layer (issues_features.gpkg)
# -----------------------------------------------------------------------------
# Create an issue registry per feature
issue_map = {idx: [] for idx in gdf.index}

# ID duplicates
if "GID_2" in gdf.columns:
    gid2 = gdf["GID_2"].astype(str)
    dup_id_mask = gid2.duplicated(keep=False)
    for idx in gdf.index[dup_id_mask]:
        issue_map[idx].append("DUPLICATE_GID_2")

# Geometry duplicates
if n_dup_geom > 0:
    for idx in gdf.index[dup_geom_mask]:
        issue_map[idx].append("DUPLICATE_GEOMETRY")

# Invalid geoms
if n_invalid > 0:
    for idx in invalid_idx:
        issue_map[idx].append("INVALID_GEOMETRY")

# Out-of-scope (expected province)
if expected_name1 and "NAME_1" in gdf.columns:
    mismatch = gdf["NAME_1"].astype(str) != expected_name1
    for idx in gdf.index[mismatch]:
        issue_map[idx].append("OUT_OF_SCOPE_NAME_1")

# Overlaps: tag both members of overlap pairs
if overlap_pairs is not None and len(overlap_pairs) > 0:
    for rec in overlap_pairs:
        issue_map[rec["i"]].append("OVERLAPS_OTHER_POLYGON")
        issue_map[rec["j"]].append("OVERLAPS_OTHER_POLYGON")

# Slivers
if ea_ok and "F3 Slivers (very small polygons)" in [c["check"] for c in checks]:
    # rebuild sliver mask in case it's needed
    areas = gdf_ea.geometry.area
    median_area = float(np.nanmedian(areas.values)) if len(areas) else 0.0
    thresh = min(0.005 * median_area, 1e6) if median_area > 0 else 0.0
    sliver_mask = areas < thresh if thresh > 0 else np.array([False] * len(gdf_ea))
    for idx in gdf.index[sliver_mask]:
        issue_map[idx].append("SLIVER_SMALL_AREA")

# Build issues GeoDataFrame (only if any issues)
flagged = [idx for idx, issues in issue_map.items() if len(issues) > 0]

if len(flagged) > 0:
    issues_gdf = gdf.loc[flagged].copy()
    issues_gdf["issues"] = [(";".join(sorted(set(issue_map[idx])))) for idx in issues_gdf.index]
    issues_gdf["issue_count"] = issues_gdf["issues"].apply(lambda x: 0 if pd.isna(x) or x == "" else len(x.split(";")))

    # Save as GeoPackage
    # Use a stable layer name
    layer_name = "issues_features"
    issues_gdf.to_file(gpkg_out, layer=layer_name, driver="GPKG")

    add_check(
        "R1 Issues GeoPackage written",
        "PASS",
        metrics={"flagged_feature_count": int(len(issues_gdf)), "gpkg": str(gpkg_out), "layer": layer_name},
        note="Wrote a GeoPackage containing only flagged features, with an 'issues' column."
    )
else:
    add_check(
        "R1 Issues GeoPackage written",
        "PASS",
        metrics={"flagged_feature_count": 0},
        note="No features flagged by QA checks; no issues GeoPackage created."
    )


# -----------------------------------------------------------------------------
# Write qa_summary.json and qa_summary.csv
# -----------------------------------------------------------------------------
summary = {
    "run_time_utc": run_time_utc,
    "boundary_path": str(boundary_path),
    "output_dir": str(output_dir),
    "expected_name1": expected_name1,
    "equal_area_crs": equal_area_crs,
    "checks": checks,
}

# CSV: one row per check with flattened metrics (key numbers)
# Keep metrics as JSON string to preserve structure without clutter
csv_rows = []
for c in checks:
    csv_rows.append({
        "check": c["check"],
        "status": c["status"],
        "note": c["note"],
        "metrics_json": json.dumps(c["metrics"], ensure_ascii=False),
    })

pd.DataFrame(csv_rows).to_csv(csv_out, index=False)
with open(json_out, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)


# -----------------------------------------------------------------------------
# Console summary: concise and interview-copyable
# -----------------------------------------------------------------------------
# Build a short list of key outcomes
status_counts = pd.Series([c["status"] for c in checks]).value_counts().to_dict()
overall = worst_status([c["status"] for c in checks if not c["check"].startswith("R")])

# Pull key signals
def find_check(name):
    for c in checks:
        if c["check"] == name:
            return c
    return None

c_gid = find_check("C4 Uniqueness of GID_2")
c_geomdup = find_check("E1 Duplicate geometries (exact)")
c_invalid = find_check("D1 Missing/empty/invalid geometries")
c_outscope = find_check("G1 Out-of-scope by expected NAME_1")
c_overlap = find_check("F1 Overlaps between polygons")
c_holes = find_check("F2 Enclosed gaps (holes inside union)")

print("\n================= BOUNDARY QA SUMMARY =================")
print("Run time (UTC):", run_time_utc)
print("Input:", boundary_path.name)
print("Overall status:", overall)
print("Status counts:", status_counts)
print("Outputs written:")
print(" -", csv_out)
print(" -", json_out)
if gpkg_out.exists():
    print(" -", gpkg_out, "(issues-only layer)")

print("\nKey findings (copy into interview notes):")
if c_gid:
    print(f"• GID_2 uniqueness: {c_gid['status']} | {c_gid['note']}")
    if c_gid["status"] != "PASS" and dup_id_csv.exists():
        print("  - details:", dup_id_csv)

if c_geomdup:
    print(f"• Duplicate geometries: {c_geomdup['status']} | {c_geomdup['note']}")
    if c_geomdup["status"] != "PASS" and dup_geom_csv.exists():
        print("  - details:", dup_geom_csv)

if c_invalid:
    print(f"• Geometry validity: {c_invalid['status']} | {c_invalid['metrics']}")

if c_outscope:
    print(f"• Out-of-scope (NAME_1 vs expected='{expected_name1}'): {c_outscope['status']} | {c_outscope['note']}")
    if c_outscope["status"] != "PASS" and outscope_csv.exists():
        print("  - details:", outscope_csv)

if c_overlap:
    print(f"• Overlaps: {c_overlap['status']} | {c_overlap['note']}")
    if c_overlap["status"] != "PASS" and overlaps_csv.exists():
        print("  - details:", overlaps_csv)

if c_holes:
    print(f"• Enclosed gaps (holes): {c_holes['status']} | {c_holes['note']}")

print("=======================================================\n")

# Cleanup temporary column
if "_geom_hash" in gdf.columns:
    gdf.drop(columns=["_geom_hash"], inplace=True)


C:\Users\am636\AppData\Local\Temp\ipykernel_8080\3650572904.py:684: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  union = gdf_ea.geometry.unary_union



================= BOUNDARY QA SUMMARY =================
Run time (UTC): 2026-01-24T23:25:52+00:00
Input: ZMB_adm2_gadm41_Copperbelt.shp
Overall status: FAIL
Status counts: {'PASS': 17, 'FAIL': 4, 'WARN': 2}
Outputs written:
 - C:\Users\am636\Desktop\Job\applied\Shortlisted\Specialist Technician_Demographic Data\Interview_package2\in_phyton\outputs\qa_summary.csv
 - C:\Users\am636\Desktop\Job\applied\Shortlisted\Specialist Technician_Demographic Data\Interview_package2\in_phyton\outputs\qa_summary.json
 - C:\Users\am636\Desktop\Job\applied\Shortlisted\Specialist Technician_Demographic Data\Interview_package2\in_phyton\outputs\issues_features.gpkg (issues-only layer)

Key findings (copy into interview notes):
• GID_2 uniqueness: FAIL | Duplicate GID_2 values found.
  - details: C:\Users\am636\Desktop\Job\applied\Shortlisted\Specialist Technician_Demographic Data\Interview_package2\in_phyton\outputs\duplicates_by_id.csv
• Duplicate geometries: FAIL | Exact duplicate geometries detected (